# Coarse-grained polyacrylamide melt: MuPT → flowerMD bead-spring DPD → MuPT → MD input files

MuPT keeps one hierarchy for a polymer, from the whole system down to atoms, and lets you pick the resolution you simulate at.
Here we build polyacrylamide (PAAm) chains atom by atom, then simulate them **one bead per repeat unit**:

| Step | Tool | What happens |
|---|---|---|
| 1–3 | MuPT | build repeat units from SMILES and link them into chains |
| 4 | MuPT → mBuild | export the hierarchy at the **repeat-unit** level: one bead per repeat unit |
| 5 | flowerMD | place the bead chains with a random walk and relax them with bead-spring DPD |
| 6 | mBuild → MuPT | write the relaxed bead positions back onto the MuPT representation |
| 7 | MuPT → MDAnalysis, HOOMD | write a coarse-grained PDB and a HOOMD `.gsd` file |

**Environment:** `conda-envs/workflows.yml` (or `pixi run -e workflows lab`). Runs in under a minute on a laptop CPU.

> This uses the refactored MuPT API that is still under review (MuPT-hub/mupt#141 through #166), so names may still change before release.

In [ ]:
import numpy as np
import networkx as nx
from networkx import connected_components, get_node_attributes, spring_layout
from networkx.drawing import draw as nx_draw

from mupt.graphs.generators import path_graph_from_sequence
from mupt.interfaces.smiles import primitive_from_smiles
from mupt.mupr.connection.alignment import ConnectorAntialignmentRigid
from mupt.mupr.primitives import CompositePrimitive, RootPrimitive
from mupt.mupr.primitives.predicates import is_superatomic
from mupt.mupr.primitives.visualize import visualize_cross_section

## 1. System size

In [ ]:
num_chains: int = 5
DOP: int = 20  # repeat units (= beads) per chain

## 2. Repeat units from SMILES

Each acrylamide repeat unit is a SMILES fragment; `[*:1]` and `[*:2]` mark where it bonds to its neighbors, and the end groups cap one side with hydrogen.

The labels `_A`, `_B` and `_C` matter: they become the **bead types** of the coarse-grained model, and they are the bead names flowerMD's `Bead_Spring_DPD` force field (step 5) has parameters for.

In [ ]:
lexicon = {
    "head": primitive_from_smiles("[H:1]-[C:2]CC(=O)[N:1](-[H])-[*:2]", label="_A", embed_positions=True),
    "mid":  primitive_from_smiles("[*:1]-[C:2]CC(=O)[N:1](-[H])-[*:2]", label="_B", embed_positions=True),
    "tail": primitive_from_smiles("[*:1]-[C:2]CC(=O)[N:1](-[H])-[H:2]", label="_C", embed_positions=True),
}
sequence = ["head"] + (DOP - 2) * ["mid"] + ["tail"]

sequence_graph = path_graph_from_sequence(sequence, label_attr="label")
nx_draw(sequence_graph, with_labels=True, labels=get_node_attributes(sequence_graph, "label"), node_size=200, font_size=7)

## 3. Link repeat units into chains

Same as in `01_pe_all_atom_dpd`: each repeat unit is placed on its neighbor's free Connector, then bonded.
The chain is tagged `role="segment"` and each repeat unit `role="residue"`; the coarse-grained exporter in step 7 turns each `residue` into one bead.

> `populate_from_topology_and_lexicon` will do this in one call once it can link repeat units that already have coordinates; until then this short loop does the same job.

In [ ]:
def build_chain(sequence, bond_length=1.54):
    # Place each embedded repeat unit on its neighbor's Connector, then bond them.
    chain = CompositePrimitive(label="PAAm", role="segment")
    units = [lexicon[name].copy() for name in sequence]
    for unit in units:
        unit.role = "residue"
        chain.attach_child(unit)

    align = ConnectorAntialignmentRigid()
    for a, b in zip(units, units[1:]):
        x, y = next(
            (x, y)
            for x in a.connections.connectors_free
            for y in b.connections.connectors_free
            if x.bondable_with(y)
        )
        x.set_bond_length(bond_length)
        y.set_bond_length(bond_length)
        b.rigidly_transform(align.antialignment_transformation(align_connector=y, to_connector=x))
        a.connect_neighbor(b, our_connector=x, their_connector=y)
    return chain


system = RootPrimitive(label="PAAm melt", role="universe")
for _ in range(num_chains):
    system.attach_child(build_chain(sequence))

n_atoms = sum(1 for _ in system.leaves)
print(f"{len(system.children)} chains, {num_chains * DOP} repeat units, {n_atoms} atoms")

## 4. Choose the coarse-grained resolution, then export it to mBuild

A **cross-section** slices the hierarchy at one level. `is_superatomic` selects every Primitive whose children are all atoms: here, the repeat units.
The result is the bead-level bond graph: one node per bead, one edge per bond between neighboring repeat units.

In [ ]:
beads = system.cross_section(is_superatomic)
print(f"{len(list(connected_components(beads)))} chains, {beads.number_of_nodes()} beads, {beads.number_of_edges()} bonds")
visualize_cross_section(system, beads, pos=spring_layout(beads, dim=2, seed=1), with_labels=False, node_size=30)

In [ ]:
from mupt.interfaces.mosdef.mbuild import primitive_to_mbuild

compound = primitive_to_mbuild(
    system,
    predicate=is_superatomic,  # repeat units become the mBuild particles (beads)
    preserve_intermediates=True,  # keep one sub-Compound per chain
    coords_to_nm=0.1,  # MuPT coordinates are in Angstrom; mBuild/flowerMD use nm
    bead_position="centroid",  # bead at its repeat unit's geometric center ("center_of_mass" also works)
)
print(compound)
print("bead names:", sorted({particle.name for particle in compound.particles()}))

## 5. Place and relax the bead chains with flowerMD

flowerMD gets one `Molecule` per MuPT chain.

- `RandomWalk` places each chain as a random walk with 1.0 nm steps (the bond length of the force field below), at 0.8 beads/nm³.
- `Bead_Spring_DPD` loads harmonic bonds (r₀ = 1.0 nm) and a soft DPD repulsion between beads, for bead types `_A`, `_B` and `_C`. Its parameters are in kJ/mol, nm and amu, so those are the base units.
- `PhantomWalk` runs DPD to push overlapping beads apart, then FIRE energy minimization, and reports the closest distance between any two beads.

In [ ]:
import warnings

import hoomd
import unyt as u
from flowermd.base import Molecule
from flowermd.library import Bead_Spring_DPD, PhantomWalk, RandomWalk

warnings.filterwarnings("ignore")

chains = [  # one flowerMD Molecule per MuPT chain
    Molecule(num_mols=1, compound=chain, identify_connections=False)
    for chain in compound.children
]
walk = RandomWalk(
    molecules=chains,
    bond_length=1.0,  # nm
    density=0.8 * u.Unit("nm**-3"),  # beads per nm^3
    base_units={"length": 1 * u.nm, "mass": 1 * u.amu, "energy": 1 * u.Unit("kJ/mol")},
    seed=456,
)
print("beads:", walk.n_particles, " box (nm):", walk.box.Lx)

nlist = hoomd.md.nlist.Cell(buffer=0.1, exclusions=["bond"])
walk.apply_forcefield(force_field=Bead_Spring_DPD(), kT=1.0, r_cut=1.01, nlist=nlist)
print("bead types:", walk.hoomd_snapshot.particles.types)

In [ ]:
sim = PhantomWalk(
    walk.hoomd_snapshot,
    walk.hoomd_forcefield,
    n_steps_dpd=5000,
    n_steps_fire=500,
    gsd_write_freq=1000,  # trajectory.gsd: watch the beads spread out
    log_write_freq=1000,
)
print(f"closest beads after DPD + FIRE: {sim.particle_separation:.3f} nm")

## 6. Put the relaxed bead positions back on the MuPT representation

HOOMD keeps every bead inside the periodic box, so a chain that crosses a face of the box comes back in pieces.
The random walk already placed the beads inside the box, so HOOMD's image counters can't undo that; instead, walk along each chain's bonds and step each bead by the shortest periodic distance from the previous one.

Then copy the whole chains onto the exported Compound and let `update_primitive_from_mbuild` move each repeat unit (bead) to its new position, by address.

In [ ]:
from mupt.interfaces.mosdef.mbuild import update_primitive_from_mbuild

snapshot = sim.state.get_snapshot()
box_nm = np.asarray(snapshot.configuration.box[:3])

bond_graph = nx.Graph(snapshot.bonds.group.tolist())
whole_nm = np.array(snapshot.particles.position, dtype=float)
for chain in connected_components(bond_graph):
    for i, j in nx.bfs_edges(bond_graph, min(chain)):
        step = whole_nm[j] - whole_nm[i]
        whole_nm[j] = whole_nm[i] + step - box_nm * np.round(step / box_nm)

compound.xyz = whole_nm  # flowerMD keeps the bead order of the exported Compound
moved = update_primitive_from_mbuild(system, compound, coords_to_nm=0.1, bead_position="centroid")
print(f"updated {len(moved)} beads on the MuPT representation")

## 7. Look at the relaxed melt

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
for chain in system.children:
    xyz = np.array([unit.shape.centroid for unit in chain.children]) / 10  # nm
    ax.plot(*xyz.T, marker="o", markersize=4)
ax.set_xlabel("x (nm)")
ax.set_ylabel("y (nm)")
ax.set_zlabel("z (nm)")
ax.set_title(f"{num_chains} PAAm chains, one bead per repeat unit")
plt.show()

## 8. Write coarse-grained files

`primitive_to_mdanalysis` with `CoarseGrainedExportStrategy` writes one bead per `residue` (repeat unit), grouped into one segment per chain; bead masses are the repeat units' masses.
Both exporters take `bead_position`: `"centroid"` (the repeat unit's geometric center, the default) or `"center_of_mass"` (its atoms' mass-weighted center). Use the same choice everywhere: with `"centroid"` here as in step 4, the beads in the PDB sit exactly where the simulation put them.

For HOOMD, write the relaxed state straight from the simulation as a `.gsd` file.

In [ ]:
from mupt.interfaces.mdanalysis import primitive_to_mdanalysis
from mupt.interfaces.mdanalysis.strategies import CoarseGrainedExportStrategy

universe = primitive_to_mdanalysis(
    system,
    {"_A": "AMH", "_B": "AMM", "_C": "AMT"},  # 3-letter names for head, middle and tail beads
    strategy=CoarseGrainedExportStrategy(bead_position="centroid"),  # same choice as the mBuild export
)
universe.dimensions = [*(box_nm * 10), 90.0, 90.0, 90.0]  # Angstrom
universe.atoms.write("paam_cg.pdb")
print(f"{universe.atoms.n_atoms} beads, {universe.segments.n_segments} chains, {len(universe.bonds)} bonds")

hoomd.write.GSD.write(state=sim.state, filename="paam_cg.gsd", mode="wb")

> **What about the atoms?** Each repeat unit's atoms moved rigidly with its bead, so they sit around the right place but are **not** a valid all-atom structure (neighboring repeat units are now 1 nm apart). Getting atomistic coordinates from a coarse-grained model needs a backmapping step, which is not covered here.